## Metadata analysis 


Date created: 10/23/2024

Notebook author: Britta De Pessemier 

Data analysis by:  Britta De Pessemier


In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

import biom
from biom import load_table
from qiime2 import Artifact
import qiime2 as q2

In [8]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_22102024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

In [9]:
# Now filter out the samples that match 'low Acne_NL'
samples_to_drop = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']

# Get the number of samples that are being dropped
num_dropped_samples = len(samples_to_drop)

# Drop the samples
metadata_df = metadata_df[metadata_df['severity_group'] != 'low Acne_NL']

# Output the number of dropped samples
print(f'Number of samples dropped: {num_dropped_samples}')

Number of samples dropped: 23


In [33]:
# Set the color palette for the groups
palette = {
    'Acne_L': "#ff676c",      # Red color for Acne Lesional
    'Acne_NL': "#17c6d5",     # Blue color for Acne Non-Lesional
    'Healthy': "#000096"      # Dark Blue color for Healthy
}

# Create a more rectangular plot (e.g., 8 inches wide and 12 inches tall)
plt.figure(figsize=(8, 12))

# Plot the skin pH across the groups using a boxplot with custom colors
ax = sns.boxplot(x='group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=palette)

# Add the strip plot (the jitter parameter adds some spread to the points for better visualization)
sns.stripplot(x='group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=palette, jitter=True, dodge=False, ax=ax, linewidth=0.6)

# Add the title and labels
plt.title('Skin pH at Forehead Right', fontsize=18)
plt.xlabel(' ')
plt.ylabel('Skin pH', fontsize=16)
# Adjust the y-axis limits to be tighter and remove excess empty space
plt.ylim(4.35, 6.2)  # Adjust the range based on your data distribution

# Custom labels for the x-axis
new_labels = ['Acne\nLesional', 'Acne\nNon-lesional', 'Healthy']
plt.xticks(ticks=range(len(new_labels)), labels=new_labels, rotation=0, ha='center', fontsize=16)

# Pairwise significance testing using Mann-Whitney U test
groups = ['Acne_L', 'Acne_NL', 'Healthy']
p_values = {}

# Heights to draw the annotation lines
y_max = 6.0  # Adjust this to start drawing brackets below the y-axis limit
height_step = 0.05  # Smaller step between brackets

# Perform pairwise comparisons
for i, group1 in enumerate(groups):
    for j, group2 in enumerate(groups):
        if i < j:
            # Get the skin pH values for each group
            group1_values = metadata_df[metadata_df['group'] == group1]['skin_ph_meter_ph_mean_forehead_right']
            group2_values = metadata_df[metadata_df['group'] == group2]['skin_ph_meter_ph_mean_forehead_right']
            
            # Perform Mann-Whitney U test
            stat, p = mannwhitneyu(group1_values, group2_values, alternative='two-sided')
            p_values[f'{group1} vs {group2}'] = p
            
            # Plot only if p-value is significant (p < 0.05)
            if p < 0.05:
                # Determine the significance label based on p-value thresholds
                if p < 0.001:
                    label = '***'
                elif p < 0.01:
                    label = '**'
                else:
                    label = '*'
                
                # Get x coordinates of the boxplots
                x1, x2 = i, j
                y = y_max + height_step  # Vertical position for the horizontal line
                
                # Draw horizontal line and annotate the significance label
                plt.plot([x1, x1, x2, x2], [y, y + 0.01, y + 0.01, y], lw=1, color='black')
                plt.text((x1 + x2) * 0.5, y + 0.02, label, ha='center', va='bottom', fontsize=12)
                
                # Update y_max for the next comparison
                y_max += height_step + 0.05

# Save the figure
plt.savefig(f'../Figures/metadata/skin_ph_forehead_right.png', dpi=600)  # Save as png
plt.savefig(f'../Figures/metadata/skin_ph_forehead_right.svg')  # Save as svg

# Print pairwise p-values in scientific notation
print("Pairwise Mann-Whitney U test p-values:")
for comparison, p_value in p_values.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Show the plot
plt.show()


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/598275293.py:15: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=palette, jitter=True, dodge=False, ax=ax, linewidth=0.6)


Pairwise Mann-Whitney U test p-values:
Acne_L vs Acne_NL: p-value = 9.26e-01
Acne_L vs Healthy: p-value = 1.54e-02
Acne_NL vs Healthy: p-value = 1.40e-01


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/598275293.py:79: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [17]:

# Define the color palette for the severity groups
severity_group_colors = {
    "absent Healthy": "#000096",       # Dark Blue for Healthy
    "absent Acne_NL": "#17c6d5",       # Light Blue for Acne Non-lesional (absent)
    "low Acne_L": "#ff676c",           # Light Red for Low Acne Lesional
    "moderate Acne_L": "#ca0000",      # Darker Red for Moderate Acne Lesional
    "high Acne_L": "#960000"           # Darkest Red for High Acne Lesional
}

# Ensure the severity groups are ordered as requested
metadata_df['severity_group'] = pd.Categorical(metadata_df['severity_group'], 
                                                categories=['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L'], 
                                                ordered=True)

# Create a more rectangular plot (e.g., 8 inches wide and 6 inches tall)
plt.figure(figsize=(8, 7))

# Plot the skin pH across the severity groups using a boxplot with custom colors
ax = sns.boxplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors)

# Add the strip plot (the jitter parameter adds some spread to the points for better visualization)
sns.stripplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)

# Add the title and labels
plt.title('Skin pH at Forehead Right by Severity Group', fontsize=18)
plt.xlabel(' ')
plt.ylabel('Skin pH', fontsize=16)

# Adjust the y-axis limits to be tighter and remove excess empty space
plt.ylim(4.35, 6.2)  # Adjust the range based on your data distribution

# Custom labels for the x-axis (rotated slightly to avoid overlap)
new_labels = ['Healthy', 'Acne Non-lesional', 'Low Acne Lesional', 'Moderate Acne Lesional', 'High Acne Lesional']
plt.xticks(ticks=range(len(new_labels)), labels=new_labels, rotation=15, ha='right', fontsize=12)

# Pairwise significance testing using Mann-Whitney U test
severity_groups = ['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L']
p_values = {}

# Heights to draw the annotation lines
y_max = 6.0  # Adjust this to start drawing brackets below the y-axis limit
height_step = 0.1  # Smaller step between brackets

# Perform pairwise comparisons and plot only significant results
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            # Get the skin pH values for each severity group
            group1_values = metadata_df[metadata_df['severity_group'] == group1]['skin_ph_meter_ph_mean_forehead_right']
            group2_values = metadata_df[metadata_df['severity_group'] == group2]['skin_ph_meter_ph_mean_forehead_right']
            
            # Perform Mann-Whitney U test
            stat, p = mannwhitneyu(group1_values, group2_values, alternative='two-sided')
            p_values[f'{group1} vs {group2}'] = p
            
            # Plot only if p-value is significant (p < 0.05)
            if p < 0.05:
                # Determine the significance label based on p-value thresholds
                if p < 0.001:
                    label = '***'
                elif p < 0.01:
                    label = '**'
                else:
                    label = '*'
                
                # Get x coordinates of the boxplots
                x1, x2 = i, j
                y = y_max  # Vertical position for the horizontal line
                
                # Draw horizontal line and annotate the significance label
                plt.plot([x1, x1, x2, x2], [y, y + 0.01, y + 0.01, y], lw=1, color='black')
                plt.text((x1 + x2) * 0.5, y + 0.02, label, ha='center', va='bottom', fontsize=12)
                
                # Update y_max for the next comparison
                y_max += height_step

# Save the figure
plt.savefig(f'../Figures/metadata/skin_ph_forehead_right_severity_fixed.png', dpi=600)  # Save as png
plt.savefig(f'../Figures/metadata/skin_ph_forehead_right_severity_fixed.svg')  # Save as svg

# Print pairwise p-values in scientific notation
print("Pairwise Mann-Whitney U test p-values:")
for comparison, p_value in p_values.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Show the plot
plt.tight_layout()
plt.show()


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/4007277498.py:22: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)


Pairwise Mann-Whitney U test p-values:
absent Healthy vs absent Acne_NL: p-value = 1.40e-01
absent Healthy vs low Acne_L: p-value = 1.06e-02
absent Healthy vs moderate Acne_L: p-value = 1.44e-01
absent Healthy vs high Acne_L: p-value = 8.49e-02
absent Acne_NL vs low Acne_L: p-value = 5.46e-01
absent Acne_NL vs moderate Acne_L: p-value = 6.58e-01
absent Acne_NL vs high Acne_L: p-value = 1.00e+00
low Acne_L vs moderate Acne_L: p-value = 1.56e-01
low Acne_L vs high Acne_L: p-value = 5.80e-01
moderate Acne_L vs high Acne_L: p-value = 5.59e-01


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/4007277498.py:88: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [25]:

# Set the color palette for the groups
palette = {
    'Acne_L': "#ff676c",      # Red color for Acne Lesional
    'Acne_NL': "#17c6d5",     # Blue color for Acne Non-Lesional
    'Healthy': "#000096"     # Green color for Healthy
}

# Map the old group names to new ones for the legend
group_name_mapping = {
    "Healthy": "Healthy",
    "Acne_L": "Acne Lesional",
    "Acne_NL": "Acne Non-lesional"
}

# Create a more rectangular plot (e.g., 8 inches wide and 12 inches tall)
plt.figure(figsize=(8, 12))

# Plot the skin pH across the groups using a boxplot with custom colors
ax = sns.boxplot(x='group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=palette)

# Add the strip plot (the jitter parameter adds some spread to the points for better visualization)
sns.stripplot(x='group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=palette, jitter=True, dodge=False, ax=ax, linewidth=0.6)

# Add the title and labels
plt.title('Sebum level at Forehead Left', fontsize=18)
plt.xlabel(' ')
plt.ylabel('Sebum Level (Forehead Left)', fontsize=16)
# Adjust the y-axis limits to be tighter and remove excess empty space
plt.ylim(30, 300)  # Adjust the range based on your data distribution

# Custom labels for the x-axis
new_labels = ['Acne\nLesional', 'Acne\nNon-lesional', 'Healthy']
plt.xticks(ticks=range(len(new_labels)), labels=new_labels, rotation=0, ha='center', fontsize=16)

# Pairwise significance testing using Mann-Whitney U test
groups = ['Acne_L', 'Acne_NL', 'Healthy']
p_values = {}

# Heights to draw the annotation lines
#y_max = max(metadata_df['skin_ph_meter_ph_mean_forehead_right']) + 0.1
# Heights to draw the annotation lines
y_max = 260.0  # Adjust this to start drawing brackets below the y-axis limit
height_step = 10  # Smaller step between brackets


# Perform pairwise comparisons
for i, group1 in enumerate(groups):
    for j, group2 in enumerate(groups):
        if i < j:
            # Get the skin pH values for each group
            group1_values = metadata_df[metadata_df['group'] == group1]['sebumeter_casual_level_mean_forehead_left']
            group2_values = metadata_df[metadata_df['group'] == group2]['sebumeter_casual_level_mean_forehead_left']
            
            # Perform Mann-Whitney U test
            stat, p = mannwhitneyu(group1_values, group2_values, alternative='two-sided')
            p_values[f'{group1} vs {group2}'] = p
            
            # Determine the significance label based on p-value thresholds
            if p >= 0.05:
                label = 'ns'
            elif p < 0.001:
                label = '***'
            elif p < 0.01:
                label = '**'
            else:
                label = '*'
            
            # Get x coordinates of the boxplots
            x1, x2 = i, j
            y = y_max + height_step  # Vertical position for the horizontal line
            
            # Draw horizontal line and annotate the significance label
            plt.plot([x1, x1, x2, x2], [y, y + 0.01, y + 0.01, y], lw=1, color='black')
            plt.text((x1 + x2) * 0.5, y + 0.02, label, ha='center', va='bottom', fontsize=12)
            
            # Update y_max for the next comparison
            y_max += height_step + 0.05

# Save the figure
plt.savefig(f'../Figures/metadata/sebum_forehead_left.png', dpi=600)  # Save as png
plt.savefig(f'../Figures/metadata/sebum_forehead_left.svg')  # Save as svg

# Print pairwise p-values in scientific notation
print("Pairwise Mann-Whitney U test p-values:")
for comparison, p_value in p_values.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Show the plot
plt.show()


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/3280028518.py:22: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=palette, jitter=True, dodge=False, ax=ax, linewidth=0.6)


Pairwise Mann-Whitney U test p-values:
Acne_L vs Acne_NL: p-value = 3.62e-01
Acne_L vs Healthy: p-value = 1.05e-10
Acne_NL vs Healthy: p-value = 2.11e-06


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/3280028518.py:89: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [39]:

# Define the color palette for the severity groups
severity_group_colors = {
    "absent Healthy": "#000096",       # Dark Blue for Healthy
    "absent Acne_NL": "#17c6d5",       # Light Blue for Acne Non-lesional (absent)
    "low Acne_L": "#ff676c",           # Light Red for Low Acne Lesional
    "moderate Acne_L": "#ca0000",      # Darker Red for Moderate Acne Lesional
    "high Acne_L": "#960000"           # Darkest Red for High Acne Lesional
}

# Ensure the severity groups are ordered as requested
metadata_df['severity_group'] = pd.Categorical(metadata_df['severity_group'], 
                                                categories=['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L'], 
                                                ordered=True)

# Create the sebum level plot
fig, ax = plt.subplots(figsize=(8, 8))

# Boxplot for sebum level
sns.boxplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, ax=ax)

# Stripplot for individual data points
sns.stripplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)

# Add the title and labels
ax.set_ylabel('Sebum Level (Forehead Left)', fontsize=16)
ax.set_title('Sebum Level by Severity Group', fontsize=18)
ax.set_xlabel('')

# Custom x-axis labels to avoid overlap
new_labels = ['Healthy', 'Acne\nNon-lesional', 'Low Acne\nLesional', 'Moderate Acne\nLesional', 'High Acne\nLesional']
ax.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)

# Dynamically calculate the y_max based on the range of the y-axis
#y_range = metadata_df['sebumeter_casual_level_mean_forehead_left'].max() - metadata_df['sebumeter_casual_level_mean_forehead_left'].min()
#y_max = metadata_df['sebumeter_casual_level_mean_forehead_left'].max() + y_range * 0.2  # Add 20% space above max
y_max = 260.0  # Adjust this to start drawing brackets below the y-axis limit
height_increment = 15 
# Function to draw significance brackets without overlap
def add_significance(ax, x1, x2, y_start, p_value, height_step):
    # Set significance label based on p-value
    if p_value >= 0.05:
        label = 'ns'
    elif p_value < 0.001:
        label = '***'
    elif p_value < 0.01:
        label = '**'
    else:
        label = '*'

    # Plot the significance brackets and labels
    y = y_start + height_step  # Increment the vertical position
    ax.plot([x1, x1, x2, x2], [y, y + 0.01 * y_range, y + 0.01 * y_range, y], lw=1, color='black')
    ax.text((x1 + x2) * 0.5, y + 0.02 * y_range, label, ha='center', va='bottom', fontsize=12)

# Pairwise significance testing for sebum level
severity_groups = ['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L']
p_values_sebum = {}

# Initial height for significance annotations
base_height = y_max  # Set base height above the max y-value
#height_increment = y_range * 0.1  # Vertical step is now 10% of the y-axis range

# Adjusted height for each pair comparison
bracket_heights = {}

# Perform pairwise comparisons and plot significant results for sebum level
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            # Get the sebum level values for each severity group
            group1_values_sebum = metadata_df[metadata_df['severity_group'] == group1]['sebumeter_casual_level_mean_forehead_left']
            group2_values_sebum = metadata_df[metadata_df['severity_group'] == group2]['sebumeter_casual_level_mean_forehead_left']
            
            # Perform Mann-Whitney U test
            stat, p_sebum = mannwhitneyu(group1_values_sebum, group2_values_sebum, alternative='two-sided')
            p_values_sebum[f'{group1} vs {group2}'] = p_sebum
            
            # Plot only if p-value is significant (p < 0.05)
            if p_sebum < 0.05:
                # Ensure each comparison has a unique height offset to avoid overlap
                comparison_key = (group1, group2)
                if comparison_key not in bracket_heights:
                    bracket_heights[comparison_key] = base_height
                    base_height += height_increment  # Increase base_height for the next comparison
                
                add_significance(ax, i, j, bracket_heights[comparison_key], p_sebum, height_increment)

# Set the y-limit to ensure space for significance brackets
ax.set_ylim(0, base_height + height_increment)

# Save the figure
plt.savefig(f'../Figures/metadata/sebum_level_significance_fixed.png', dpi=600)  # Save as png
plt.savefig(f'../Figures/metadata/sebum_level_significance_fixed.svg')  # Save as svg

# Print pairwise p-values in scientific notation for sebum level
print("Pairwise Mann-Whitney U test p-values (sebum level):")
for comparison, p_value in p_values_sebum.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Show the plot
plt.tight_layout()
plt.show()


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/1471772670.py:22: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax, linewidth=0.6)


Pairwise Mann-Whitney U test p-values (sebum level):
absent Healthy vs absent Acne_NL: p-value = 2.11e-06
absent Healthy vs low Acne_L: p-value = 2.58e-09
absent Healthy vs moderate Acne_L: p-value = 1.45e-07
absent Healthy vs high Acne_L: p-value = 1.71e-04
absent Acne_NL vs low Acne_L: p-value = 4.59e-01
absent Acne_NL vs moderate Acne_L: p-value = 5.63e-01
absent Acne_NL vs high Acne_L: p-value = 1.64e-01
low Acne_L vs moderate Acne_L: p-value = 9.04e-01
low Acne_L vs high Acne_L: p-value = 2.57e-01
moderate Acne_L vs high Acne_L: p-value = 2.33e-01


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/1471772670.py:102: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [20]:


# Define the color palette for the severity groups
severity_group_colors = {
    "absent Healthy": "#000096",       # Dark Blue for Healthy
    "absent Acne_NL": "#17c6d5",       # Light Blue for Acne Non-lesional (absent)
    "low Acne_L": "#ff676c",           # Light Red for Low Acne Lesional
    "moderate Acne_L": "#ca0000",      # Darker Red for Moderate Acne Lesional
    "high Acne_L": "#960000"           # Darkest Red for High Acne Lesional
}

# Ensure the severity groups are ordered as requested
metadata_df['severity_group'] = pd.Categorical(metadata_df['severity_group'], 
                                                categories=['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L'], 
                                                ordered=True)

# Create two subplots next to each other but without sharing the y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# First subplot for skin pH (ax1)
sns.boxplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, ax=ax1)
sns.stripplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
ax1.set_ylabel('Skin pH (Forehead Right)', fontsize=16)
ax1.set_xlabel('')  # No x-axis label
ax1.set_title('Skin pH by Severity Group', fontsize=18)

# Second subplot for sebum level (ax2)
sns.boxplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, ax=ax2)
sns.stripplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax2, linewidth=0.6)
ax2.set_ylabel('Sebum Level (Forehead Left)', fontsize=16)
ax2.set_xlabel('')  # No x-axis label
ax2.set_title('Sebum Level by Severity Group', fontsize=18)

# Custom x-axis labels to avoid overlap
new_labels = ['Healthy', 'Acne\nNon-lesional', 'Low Acne\nLesional', 'Moderate Acne\nLesional', 'High Acne\nLesional']
ax1.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)
ax2.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)

# Function to draw significance brackets closer to the boxplots and one another
def add_significance(ax, x1, x2, y_start, p_value, height_step, significance_offset):
    # Set significance label
    if p_value >= 0.05:
        label = 'ns'
    elif p_value < 0.001:
        label = '***'
    elif p_value < 0.01:
        label = '**'
    else:
        label = '*'

    # Draw the significance brackets
    y = y_start + height_step  # Vertical position for the horizontal line
    ax.plot([x1, x1, x2, x2], [y, y + significance_offset, y + significance_offset, y], lw=1, color='black')
    ax.text((x1 + x2) * 0.5, y + significance_offset * 0.75, label, ha='center', va='bottom', fontsize=12)

# Pairwise significance testing for the skin pH plot (ax1)
severity_groups = ['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L']
p_values_ph = {}

# Heights to draw the annotation lines for skin pH
y_max_ph = max(metadata_df['skin_ph_meter_ph_mean_forehead_right']) + 0.05  # Reduce initial height for closer brackets
height_step_ph = 0.05  # Reduce the height between brackets
significance_offset = 0.01  # Bring the significance label closer to the bracket

# Perform pairwise comparisons and plot only significant results for skin pH
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            # Get the skin pH values for each severity group
            group1_values_ph = metadata_df[metadata_df['severity_group'] == group1]['skin_ph_meter_ph_mean_forehead_right']
            group2_values_ph = metadata_df[metadata_df['severity_group'] == group2]['skin_ph_meter_ph_mean_forehead_right']
            
            # Perform Mann-Whitney U test
            stat, p_ph = mannwhitneyu(group1_values_ph, group2_values_ph, alternative='two-sided')
            p_values_ph[f'{group1} vs {group2}'] = p_ph
            
            # Plot only if p-value is significant (p < 0.05)
            if p_ph < 0.05:
                add_significance(ax1, i, j, y_max_ph, p_ph, height_step_ph, significance_offset)
                y_max_ph += height_step_ph  # Increment height after each bracket

# Pairwise significance testing for the sebum level plot (ax2)
p_values_sebum = {}

# Heights to draw the annotation lines for sebum level
y_max_sebum = max(metadata_df['sebumeter_casual_level_mean_forehead_left']) + 0.05  # Reduce initial height
height_step_sebum = 10  # Adjust the height between brackets based on the sebum data's larger range

# Perform pairwise comparisons and plot only significant results for sebum level
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            # Get the sebum level values for each severity group
            group1_values_sebum = metadata_df[metadata_df['severity_group'] == group1]['sebumeter_casual_level_mean_forehead_left']
            group2_values_sebum = metadata_df[metadata_df['severity_group'] == group2]['sebumeter_casual_level_mean_forehead_left']
            
            # Perform Mann-Whitney U test
            stat, p_sebum = mannwhitneyu(group1_values_sebum, group2_values_sebum, alternative='two-sided')
            p_values_sebum[f'{group1} vs {group2}'] = p_sebum
            
            # Plot only if p-value is significant (p < 0.05)
            if p_sebum < 0.05:
                add_significance(ax2, i, j, y_max_sebum, p_sebum, height_step_sebum, significance_offset)
                y_max_sebum += height_step_sebum  # Increment height after each bracket

# Set different y-limits for the two subplots to avoid overlap of significance brackets
ax1.set_ylim(4.35, 6.1)  # Skin pH range
ax2.set_ylim(0, y_max_sebum + height_step_sebum)  # Sebum level range

# Save the figure
plt.savefig(f'../Figures/metadata/skin_ph_sebum_forehead_side_by_side_fixed_yaxis.png', dpi=600)  # Save as png
plt.savefig(f'../Figures/metadata/skin_ph_sebum_forehead_side_by_side_fixed_yaxis.svg')  # Save as svg

# Print pairwise p-values in scientific notation for skin pH
print("Pairwise Mann-Whitney U test p-values (skin pH):")
for comparison, p_value in p_values_ph.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Print pairwise p-values in scientific notation for sebum level
print("Pairwise Mann-Whitney U test p-values (sebum level):")
for comparison, p_value in p_values_sebum.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Show the plot
plt.tight_layout()
plt.show()


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/2257782433.py:24: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/2257782433.py:31: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax2, linewidth=0.6)


Pairwise Mann-Whitney U test p-values (skin pH):
absent Healthy vs absent Acne_NL: p-value = 1.40e-01
absent Healthy vs low Acne_L: p-value = 1.06e-02
absent Healthy vs moderate Acne_L: p-value = 1.44e-01
absent Healthy vs high Acne_L: p-value = 8.49e-02
absent Acne_NL vs low Acne_L: p-value = 5.46e-01
absent Acne_NL vs moderate Acne_L: p-value = 6.58e-01
absent Acne_NL vs high Acne_L: p-value = 1.00e+00
low Acne_L vs moderate Acne_L: p-value = 1.56e-01
low Acne_L vs high Acne_L: p-value = 5.80e-01
moderate Acne_L vs high Acne_L: p-value = 5.59e-01
Pairwise Mann-Whitney U test p-values (sebum level):
absent Healthy vs absent Acne_NL: p-value = 2.11e-06
absent Healthy vs low Acne_L: p-value = 2.58e-09
absent Healthy vs moderate Acne_L: p-value = 1.45e-07
absent Healthy vs high Acne_L: p-value = 1.71e-04
absent Acne_NL vs low Acne_L: p-value = 4.59e-01
absent Acne_NL vs moderate Acne_L: p-value = 5.63e-01
absent Acne_NL vs high Acne_L: p-value = 1.64e-01
low Acne_L vs moderate Acne_L: p-

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/2257782433.py:128: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [ ]:


# Define the color palette for the severity groups
severity_group_colors = {
    "absent Healthy": "#000096",       # Dark Blue for Healthy
    "absent Acne_NL": "#17c6d5",       # Light Blue for Acne Non-lesional (absent)
    "low Acne_L": "#ff676c",           # Light Red for Low Acne Lesional
    "moderate Acne_L": "#ca0000",      # Darker Red for Moderate Acne Lesional
    "high Acne_L": "#960000"           # Darkest Red for High Acne Lesional
}

# Ensure the severity groups are ordered as requested
metadata_df['severity_group'] = pd.Categorical(metadata_df['severity_group'], 
                                                categories=['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L'], 
                                                ordered=True)

# Create two subplots next to each other but without sharing the y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# First subplot for skin pH (ax1)
sns.boxplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, ax=ax1)
sns.stripplot(x='severity_group', y='skin_ph_meter_ph_mean_forehead_right', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
ax1.set_ylabel('Skin pH (Forehead Right)', fontsize=16)
ax1.set_xlabel('')  # No x-axis label
ax1.set_title('Skin pH by Severity Group', fontsize=18)

# Second subplot for sebum level (ax2)
sns.boxplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, ax=ax2)
sns.stripplot(x='severity_group', y='sebumeter_casual_level_mean_forehead_left', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax2, linewidth=0.6)
ax2.set_ylabel('Sebum Level (Forehead Left)', fontsize=16)
ax2.set_xlabel('')  # No x-axis label
ax2.set_title('Sebum Level by Severity Group', fontsize=18)

# Custom x-axis labels to avoid overlap
new_labels = ['Healthy', 'Acne\nNon-lesional', 'Low Acne\nLesional', 'Moderate Acne\nLesional', 'High Acne\nLesional']
ax1.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)
ax2.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)

# Function to draw significance brackets closer to the boxplots and one another
def add_significance(ax, x1, x2, y_start, p_value, height_step, significance_offset):
    # Set significance label
    if p_value >= 0.05:
        label = 'ns'
    elif p_value < 0.001:
        label = '***'
    elif p_value < 0.01:
        label = '**'
    else:
        label = '*'

    # Draw the significance brackets
    y = y_start + height_step  # Vertical position for the horizontal line
    ax.plot([x1, x1, x2, x2], [y, y + significance_offset, y + significance_offset, y], lw=1, color='black')
    ax.text((x1 + x2) * 0.5, y + significance_offset * 0.75, label, ha='center', va='bottom', fontsize=12)

# Pairwise significance testing for the skin pH plot (ax1)
severity_groups = ['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L']
p_values_ph = {}

# Heights to draw the annotation lines for skin pH
y_max_ph = max(metadata_df['skin_ph_meter_ph_mean_forehead_right']) + 0.05  # Reduce initial height for closer brackets
height_step_ph = 0.05  # Reduce the height between brackets
significance_offset = 0.01  # Bring the significance label closer to the bracket

# Perform pairwise comparisons and plot only significant results for skin pH
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            # Get the skin pH values for each severity group
            group1_values_ph = metadata_df[metadata_df['severity_group'] == group1]['skin_ph_meter_ph_mean_forehead_right']
            group2_values_ph = metadata_df[metadata_df['severity_group'] == group2]['skin_ph_meter_ph_mean_forehead_right']
            
            # Perform Mann-Whitney U test
            stat, p_ph = mannwhitneyu(group1_values_ph, group2_values_ph, alternative='two-sided')
            p_values_ph[f'{group1} vs {group2}'] = p_ph
            
            # Plot only if p-value is significant (p < 0.05)
            if p_ph < 0.05:
                add_significance(ax1, i, j, y_max_ph, p_ph, height_step_ph, significance_offset)
                y_max_ph += height_step_ph  # Increment height after each bracket

# Pairwise significance testing for the sebum level plot (ax2)
p_values_sebum = {}

# Heights to draw the annotation lines for sebum level
y_max_sebum = max(metadata_df['sebumeter_casual_level_mean_forehead_left']) + 0.05  # Reduce initial height
height_step_sebum = 10  # Adjust the height between brackets based on the sebum data's larger range

# Perform pairwise comparisons and plot only significant results for sebum level
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            # Get the sebum level values for each severity group
            group1_values_sebum = metadata_df[metadata_df['severity_group'] == group1]['sebumeter_casual_level_mean_forehead_left']
            group2_values_sebum = metadata_df[metadata_df['severity_group'] == group2]['sebumeter_casual_level_mean_forehead_left']
            
            # Perform Mann-Whitney U test
            stat, p_sebum = mannwhitneyu(group1_values_sebum, group2_values_sebum, alternative='two-sided')
            p_values_sebum[f'{group1} vs {group2}'] = p_sebum
            
            # Plot only if p-value is significant (p < 0.05)
            if p_sebum < 0.05:
                add_significance(ax2, i, j, y_max_sebum, p_sebum, height_step_sebum, significance_offset)
                y_max_sebum += height_step_sebum  # Increment height after each bracket

# Set different y-limits for the two subplots to avoid overlap of significance brackets
ax1.set_ylim(4.35, 6.1)  # Skin pH range
ax2.set_ylim(0, y_max_sebum + height_step_sebum)  # Sebum level range

# Save the figure
plt.savefig(f'../Figures/metadata/skin_ph_sebum_forehead_side_by_side_fixed_yaxis.png', dpi=600)  # Save as png
plt.savefig(f'../Figures/metadata/skin_ph_sebum_forehead_side_by_side_fixed_yaxis.svg')  # Save as svg

# Print pairwise p-values in scientific notation for skin pH
print("Pairwise Mann-Whitney U test p-values (skin pH):")
for comparison, p_value in p_values_ph.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Print pairwise p-values in scientific notation for sebum level
print("Pairwise Mann-Whitney U test p-values (sebum level):")
for comparison, p_value in p_values_sebum.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Show the plot
plt.tight_layout()
plt.show()


# healthy vs acne

In [12]:
# Combine Acne_NL and Acne_L groups into a single 'Acne' group in the metadata
metadata_df['severity_group_combined'] = metadata_df['severity_group'].replace({
    'absent Acne_NL': 'Acne',
    'low Acne_L': 'Acne',
    'moderate Acne_L': 'Acne',
    'high Acne_L': 'Acne',
    'absent Healthy': 'Healthy'
})

# Define the color palette for the two groups: Healthy and Acne
severity_group_colors_combined = {
    "Healthy": "#423fa6",  # Dark Blue for Healthy
    "Acne": "#b11919"      # Red for combined Acne group
}

# Ensure the groups are ordered properly
metadata_df['severity_group_combined'] = pd.Categorical(metadata_df['severity_group_combined'], 
                                                        categories=['Healthy', 'Acne'], 
                                                        ordered=True)

# Create two subplots next to each other but without sharing the y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# First subplot for skin pH (ax1)
sns.boxplot(x='severity_group_combined', y='skin_ph_meter_ph_mean_forehead_right', 
            data=metadata_df, palette=severity_group_colors_combined, ax=ax1)
sns.stripplot(x='severity_group_combined', y='skin_ph_meter_ph_mean_forehead_right', 
              data=metadata_df, palette=severity_group_colors_combined, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
ax1.set_ylabel('Skin pH (Forehead Right)', fontsize=16)
ax1.set_xlabel('')  # No x-axis label
ax1.set_title('Skin pH by Group', fontsize=18)

# Second subplot for sebum level (ax2)
sns.boxplot(x='severity_group_combined', y='sebumeter_casual_level_mean_forehead_left', 
            data=metadata_df, palette=severity_group_colors_combined, ax=ax2)
sns.stripplot(x='severity_group_combined', y='sebumeter_casual_level_mean_forehead_left', 
              data=metadata_df, palette=severity_group_colors_combined, jitter=True, dodge=False, ax=ax2, linewidth=0.6)
ax2.set_ylabel('Sebum Level (Forehead Left)', fontsize=16)
ax2.set_xlabel('')  # No x-axis label
ax2.set_title('Sebum Level by Group', fontsize=18)

# Set the x-axis labels
ax1.set_xticklabels(['Healthy', 'Acne'], rotation=0, ha='right', fontsize=12)
ax2.set_xticklabels(['Healthy', 'Acne'], rotation=0, ha='right', fontsize=12)

# Perform Mann-Whitney U test to compare Healthy vs Acne for both skin pH and sebum level
p_values_combined = {}

# Get the values for skin pH and sebum levels
healthy_values_ph = metadata_df[metadata_df['severity_group_combined'] == 'Healthy']['skin_ph_meter_ph_mean_forehead_right']
acne_values_ph = metadata_df[metadata_df['severity_group_combined'] == 'Acne']['skin_ph_meter_ph_mean_forehead_right']

# Perform Mann-Whitney U test for skin pH
stat_ph, p_ph = mannwhitneyu(healthy_values_ph, acne_values_ph, alternative='two-sided')
p_values_combined['Healthy vs Acne (Skin pH)'] = p_ph

# Get the values for sebum level
healthy_values_sebum = metadata_df[metadata_df['severity_group_combined'] == 'Healthy']['sebumeter_casual_level_mean_forehead_left']
acne_values_sebum = metadata_df[metadata_df['severity_group_combined'] == 'Acne']['sebumeter_casual_level_mean_forehead_left']

# Perform Mann-Whitney U test for sebum level
stat_sebum, p_sebum = mannwhitneyu(healthy_values_sebum, acne_values_sebum, alternative='two-sided')
p_values_combined['Healthy vs Acne (Sebum Level)'] = p_sebum

# Display p-values on the plot
def add_significance_label(ax, x1, x2, p_value, y_start):
    if p_value >= 0.05:
        label = 'ns'
    elif p_value < 0.001:
        label = '***'
    elif p_value < 0.01:
        label = '**'
    else:
        label = '*'

    ax.text((x1 + x2) * 0.5, y_start, label, ha='center', va='bottom', fontsize=12)

# Add significance labels for the skin pH plot
add_significance_label(ax1, 0, 1, p_ph, max(metadata_df['skin_ph_meter_ph_mean_forehead_right']) + 0.05)

# Add significance labels for the sebum level plot
add_significance_label(ax2, 0, 1, p_sebum, max(metadata_df['sebumeter_casual_level_mean_forehead_left']) + 10)

# Set y-limits for better readability
ax1.set_ylim(4.35, 6.1)  # Adjust for skin pH
ax2.set_ylim(0, max(metadata_df['sebumeter_casual_level_mean_forehead_left']) + 20)  # Adjust for sebum level

# Save the figure
plt.savefig(f'../Figures/metadata/skin_ph_sebum_Healthy_vs_Acne.png', dpi=600)
plt.savefig(f'../Figures/metadata/skin_ph_sebum_Healthy_vs_Acne.svg')

# Print Mann-Whitney U test p-values
print(f"Healthy vs Acne (Skin pH): p-value = {p_ph:.2e}")
print(f"Healthy vs Acne (Sebum Level): p-value = {p_sebum:.2e}")

# Show the plot
plt.tight_layout()
plt.show()


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_82101/1541224620.py:27: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group_combined', y='skin_ph_meter_ph_mean_forehead_right',
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_82101/1541224620.py:36: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group_combined', y='sebumeter_casual_level_mean_forehead_left',


Healthy vs Acne (Skin pH): p-value = 1.45e-02
Healthy vs Acne (Sebum Level): p-value = 2.08e-11


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_82101/1541224620.py:98: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [42]:
# Define the color palette for the severity groups
severity_group_colors = {
    "absent Healthy": "#000096",       # Dark Blue for Healthy
    "absent Acne_NL": "#17c6d5",       # Light Blue for Acne Non-lesional (absent)
    "low Acne_L": "#ff676c",           # Light Red for Low Acne Lesional
    "moderate Acne_L": "#ca0000",      # Darker Red for Moderate Acne Lesional
    "high Acne_L": "#960000"           # Darkest Red for High Acne Lesional
}

# Ensure the severity groups are ordered as requested
metadata_df['severity_group'] = pd.Categorical(metadata_df['severity_group'], 
                                                categories=['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L'], 
                                                ordered=True)

# Create two subplots next to each other but without sharing the y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# First subplot for inflammatory lesions (ax1)
sns.boxplot(x='severity_group', y='inflammatory_lesions_face', data=metadata_df, palette=severity_group_colors, ax=ax1)
sns.stripplot(x='severity_group', y='inflammatory_lesions_face', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
ax1.set_ylabel('# Inflammatory lesions ', fontsize=16)
ax1.set_xlabel('')  # No x-axis label
ax1.set_title('# Inflammatory lesions on the face', fontsize=18)

# Second subplot for noninflammatory lesions (ax2)
sns.boxplot(x='severity_group', y='noninflammatory_lesions_face', data=metadata_df, palette=severity_group_colors, ax=ax2)
sns.stripplot(x='severity_group', y='noninflammatory_lesions_face', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax2, linewidth=0.6)
ax2.set_ylabel('# Noninflammatory lesions', fontsize=16)
ax2.set_xlabel('')  # No x-axis label
ax2.set_title('# Noninflammatory lesions on the face', fontsize=18)

# Custom x-axis labels to avoid overlap
new_labels = ['Healthy', 'Acne\nNon-lesional', 'Low Acne\nLesional', 'Moderate Acne\nLesional', 'High Acne\nLesional']
ax1.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)
ax2.set_xticklabels(new_labels, rotation=20, ha='right', fontsize=12)

# Function to draw significance brackets closer to the boxplots and one another
def add_significance(ax, x1, x2, y_start, p_value, height_step, significance_offset):
    # Set significance label
    if p_value >= 0.05:
        label = 'ns'
    elif p_value < 0.001:
        label = '***'
    elif p_value < 0.01:
        label = '**'
    else:
        label = '*'

    # Draw the significance brackets
    y = y_start + height_step  # Vertical position for the horizontal line
    ax.plot([x1, x1, x2, x2], [y, y + significance_offset, y + significance_offset, y], lw=1, color='black')
    ax.text((x1 + x2) * 0.5, y + significance_offset * 0.75, label, ha='center', va='bottom', fontsize=12)

# Pairwise significance testing for the inflammatory lesions plot (ax1)
severity_groups = ['absent Healthy', 'absent Acne_NL', 'low Acne_L', 'moderate Acne_L', 'high Acne_L']
p_values_inflammatory = {}

# Heights to draw the annotation lines for inflammatory lesions
y_max_ph = max(metadata_df['inflammatory_lesions_face']) + 0.05  # Reduce initial height for closer brackets
height_step_ph = 0.05  # Reduce the height between brackets
significance_offset = 0.01  # Bring the significance label closer to the bracket

# Perform pairwise comparisons and plot only significant results for inflammatory lesions
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            group1_values = metadata_df[metadata_df['severity_group'] == group1]['inflammatory_lesions_face']
            group2_values = metadata_df[metadata_df['severity_group'] == group2]['inflammatory_lesions_face']
            stat, p_value = mannwhitneyu(group1_values, group2_values, alternative='two-sided')
            p_values_inflammatory[f'{group1} vs {group2}'] = p_value
            
            if p_value < 0.05:
                add_significance(ax1, i, j, y_max_ph, p_value, height_step_ph, significance_offset)
                y_max_ph += height_step_ph  # Increment height after each bracket

# Pairwise significance testing for the noninflammatory lesions plot (ax2)
p_values_noninflammatory = {}

# Heights to draw the annotation lines for noninflammatory lesions
y_max_sebum = max(metadata_df['noninflammatory_lesions_face']) + 0.05  # Reduce initial height
height_step_sebum = 10  # Adjust the height between brackets based on the noninflammatory lesions

# Perform pairwise comparisons and plot only significant results for noninflammatory lesions
for i, group1 in enumerate(severity_groups):
    for j, group2 in enumerate(severity_groups):
        if i < j:
            group1_values = metadata_df[metadata_df['severity_group'] == group1]['noninflammatory_lesions_face']
            group2_values = metadata_df[metadata_df['severity_group'] == group2]['noninflammatory_lesions_face']
            stat, p_value = mannwhitneyu(group1_values, group2_values, alternative='two-sided')
            p_values_noninflammatory[f'{group1} vs {group2}'] = p_value
            
            if p_value < 0.05:
                add_significance(ax2, i, j, y_max_sebum, p_value, height_step_sebum, significance_offset)
                y_max_sebum += height_step_sebum  # Increment height after each bracket

# Save the figure
plt.savefig(f'../Figures/metadata/inflammatory_noninflammatory_lesions_face.png', dpi=600)  # Save as png
plt.savefig(f'../Figures/metadata/inflammatory_noninflammatory_lesions_face.svg')  # Save as svg

# Print pairwise p-values in scientific notation for inflammatory lesions
print("Pairwise Mann-Whitney U test p-values (inflammatory lesions):")
for comparison, p_value in p_values_inflammatory.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Print pairwise p-values in scientific notation for noninflammatory lesions
print("Pairwise Mann-Whitney U test p-values (noninflammatory lesions):")
for comparison, p_value in p_values_noninflammatory.items():
    print(f"{comparison}: p-value = {p_value:.2e}")

# Show the plot
plt.tight_layout()
plt.show()


/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/4016317846.py:20: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group', y='inflammatory_lesions_face', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax1, linewidth=0.6)
/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/4016317846.py:27: FutureWarning: Passing `palette` without assigning `hue` is deprecated.
  sns.stripplot(x='severity_group', y='noninflammatory_lesions_face', data=metadata_df, palette=severity_group_colors, jitter=True, dodge=False, ax=ax2, linewidth=0.6)


Pairwise Mann-Whitney U test p-values (inflammatory lesions):
absent Healthy vs absent Acne_NL: p-value = 4.72e-18
absent Healthy vs low Acne_L: p-value = 1.35e-08
absent Healthy vs moderate Acne_L: p-value = 1.95e-11
absent Healthy vs high Acne_L: p-value = 8.54e-06
absent Acne_NL vs low Acne_L: p-value = 2.24e-05
absent Acne_NL vs moderate Acne_L: p-value = 1.52e-03
absent Acne_NL vs high Acne_L: p-value = 2.33e-03
low Acne_L vs moderate Acne_L: p-value = 1.59e-01
low Acne_L vs high Acne_L: p-value = 7.58e-01
moderate Acne_L vs high Acne_L: p-value = 2.69e-01
Pairwise Mann-Whitney U test p-values (noninflammatory lesions):
absent Healthy vs absent Acne_NL: p-value = 4.69e-18
absent Healthy vs low Acne_L: p-value = 1.35e-08
absent Healthy vs moderate Acne_L: p-value = 1.95e-11
absent Healthy vs high Acne_L: p-value = 8.55e-06
absent Acne_NL vs low Acne_L: p-value = 7.28e-04
absent Acne_NL vs moderate Acne_L: p-value = 3.41e-04
absent Acne_NL vs high Acne_L: p-value = 4.58e-08
low Acne

/var/folders/g4/bhmr4hbn3pn5zc7fy0ffk9gh0000gp/T/ipykernel_78698/4016317846.py:112: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


# Filter Metadata file 
Drop all Acne_NL samples that have a score bigger than 0

In [51]:
# Read the metadata file
metadata_file = '../Metadata/metadata_final_22102024.tsv'
metadata_df = pd.read_csv(metadata_file, sep='\t')

# Now filter out the samples that match 'low Acne_NL'
samples_to_drop = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']

# Get the number of samples that are being dropped
num_dropped_samples = len(samples_to_drop)

# First, filter the metadata to get the samples to be removed
samples_to_remove = metadata_df[metadata_df['severity_group'] == 'low Acne_NL']['SampleID'].tolist()

# Drop the samples
metadata_df = metadata_df[metadata_df['severity_group'] != 'low Acne_NL']

# Output the number of dropped samples
print(f'Number of samples dropped: {num_dropped_samples}')

Number of samples dropped: 23


In [60]:
print(metadata_df)

              SampleID c_zone  \
0    LAMI.RD308.D16.C1     C1   
1    LAMI.RD310.D21.C1     C1   
2    LAMI.RD305.D21.C3     C3   
3    LAMI.RD306.D18.C2     C2   
4     LAMI.RD306.D7.C2     C2   
..                 ...    ...   
261  LAMI.RD317.D21.C1     C1   
262   LAMI.RD001.D0.C1     C1   
263  LAMI.RD014.D14.C2     C2   
264   LAMI.RD314.D0.C1     C1   
265  LAMI.RD001.D14.C2     C2   

    visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face  \
0                                       not applicable                  
1                                                   72                  
2                                                   69                  
3                                       not applicable                  
4                                                   90                  
..                                                 ...                  
261                                                 77                  
262                

In [62]:
metadata_df.to_csv('../Metadata/metadata_filtered_10232023.tsv', sep='\t', index=False)

In [13]:
# Function to filter the biom table based on the samples to keep
def filter_biom_table(biom_table, samples_to_remove):
    # Get all sample IDs in the biom table
    all_samples = biom_table.ids(axis='sample')
    
    # Filter the samples, excluding the ones to be removed
    samples_to_keep = [sample for sample in all_samples if sample not in samples_to_remove]
    
    # Subset the biom table to only include the samples to keep
    filtered_biom_table = biom_table.filter(samples_to_keep, axis='sample', inplace=False)
    
    return filtered_biom_table, all_samples, samples_to_keep

In [53]:
# Define the folder names, titles, and file paths for the two tables
datasets = [
    ('174951', 'V4', '../Data/16S/RPCA/174951_rarefied_table_unannotated_absolute_abundances.biom'),
    ('174950', 'V1-V3', '../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')
]

In [18]:
biom_tbl = biom.load_table('../Data/16S/RPCA/174950_rarefied_table_unannotated_absolute_abundances.biom')

In [19]:
rclr_transformed_df = pd.DataFrame(biom_tbl.matrix_data.toarray(), 
                index=biom_tbl.ids(axis='observation'), 
                columns=biom_tbl.ids(axis='sample'))

In [20]:
print(rclr_transformed_df)

                                                    LAMI.RD001.D0.C1  \
GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG...              11.0   
AGCGAACGCTGGCGGCAGGCTTAACACATGCAAGTCGAGCGCCGTAG...               0.0   
ATTGAACGCTGGCGGCATGCCTTACACATGCAAGTCGAACGGTAACA...               0.0   
GATGAACGCTAGCGATAGGCTTAACACATGCAAGTCGAGGGGCAGCA...             100.0   
GACGAACGCTGGCGGCGCGCCTAACACATGCAAGTCGAACGGAGCTT...               0.0   
...                                                              ...   
GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGCTGAAG...               5.0   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGCTGAAG...               0.0   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAGCGGAAAGG...               0.0   
GACGAACGCTGGCGGCGTGCCTAATACATGCAAGTAGAACGCTGAAG...              39.0   
GACGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGTAAGG...              24.0   

                                                    LAMI.RD001.D0.C2  \
GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGG...             

In [59]:
# Loop over both tables
for dataset_id, region_title, biom_file in datasets:
    print(f"Processing dataset: {dataset_id} - {region_title}")
    
    # Load the biom table
    biom_tbl = biom.load_table(biom_file)
    
    # Filter the biom table to remove unwanted samples
    biom_tbl_filtered, original_samples, filtered_samples = filter_biom_table(biom_tbl, samples_to_remove)
        # Check the samples that were removed
    dropped_samples = set(original_samples) - set(filtered_samples)
    
    print(f"Number of samples in original table: {len(original_samples)}")
    print(f"Number of samples after filtering: {len(filtered_samples)}")
    print(f"Samples that were removed: {dropped_samples}")

    # Transpose the filtered biom table
    biom_tbl_filtered_T = biom_tbl_filtered.transpose()

    # Convert the transposed BIOM table to a QIIME 2 Artifact
    biom_tbl_filtered_artifact = q2.Artifact.import_data('FeatureTable[Frequency]', biom_tbl_filtered_T)
    
    # Save the artifact as a .qza file
    biom_tbl_filtered_artifact.save(f'../Data/16S/CTF/filtered_{dataset_id}.qza')

Processing dataset: 174951 - V4
Number of samples in original table: 217
Number of samples after filtering: 199
Samples that were removed: {'LAMI.RD306.D28.C3', 'LAMI.RD317.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD308.D9.C3', 'LAMI.RD319.D7.C3', 'LAMI.RD314.D28.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD310.D21.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD310.D7.C3', 'LAMI.RD319.D14.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD304.D7.C3'}
Processing dataset: 174950 - V1-V3
Number of samples in original table: 236
Number of samples after filtering: 215
Samples that were removed: {'LAMI.RD317.D28.C3', 'LAMI.RD304.D28.C3', 'LAMI.RD314.D28.C3', 'LAMI.RD306.D28.C3', 'LAMI.RD310.D0.C3', 'LAMI.RD314.D14.C3', 'LAMI.RD318.D28.C3', 'LAMI.RD310.D14.C3', 'LAMI.RD308.D21.C3', 'LAMI.RD306.D0.C3', 'LAMI.RD319.D28.C3', 'LAMI.RD314.D21.C3', 'LAMI.RD318.D9.C3', 'LAMI.RD317.D21.C3', 'LAMI.RD319.D21.C3', 'LAMI.RD310.D28.C3', 'LAMI.RD308.

In [58]:
print(biom_tbl_filtered)

# Constructed from biom file
#OTU ID	LAMI.RD001.D0.C1	LAMI.RD001.D0.C2	LAMI.RD001.D14.C1	LAMI.RD001.D14.C2	LAMI.RD001.D28.C1	LAMI.RD002.D0.C2	LAMI.RD003.D14.C1	LAMI.RD002.D14.C1	LAMI.RD003.D28.C1	LAMI.RD001.D28.C2	LAMI.RD007.D0.C1	LAMI.RD003.D28.C2	LAMI.RD002.D14.C2	LAMI.RD006.D0.C1	LAMI.RD011.D28.C1	LAMI.RD007.D14.C1	LAMI.RD002.D0.C1	LAMI.RD002.D28.C1	LAMI.RD006.D0.C2	LAMI.RD007.D14.C2	LAMI.RD002.D28.C2	LAMI.RD004.D0.C1	LAMI.RD006.D14.C2	LAMI.RD304.D14.C2	LAMI.RD018.D28.C1	LAMI.RD017.D0.C2	LAMI.RD014.D0.C1	LAMI.RD004.D0.C2	LAMI.RD007.D28.C1	LAMI.RD006.D14.C1	LAMI.RD304.D14.C3	LAMI.RD003.D0.C1	LAMI.RD014.D0.C2	LAMI.RD004.D28.C1	LAMI.RD006.D28.C1	LAMI.RD305.D14.C1	LAMI.RD007.D28.C2	LAMI.RD018.D28.C2	LAMI.RD304.D4.C2	LAMI.RD014.D14.C2	LAMI.RD304.D16.C1	LAMI.RD017.D14.C1	LAMI.RD004.D28.C2	LAMI.RD014.D28.C1	LAMI.RD304.D7.C1	LAMI.RD304.D18.C2	LAMI.RD304.D0.C1	LAMI.RD011.D0.C1	LAMI.RD305.D14.C2	LAMI.RD017.D28.C2	LAMI.RD014.D28.C2	LAMI.RD306.D21.C2	LAMI.RD304.D9.C1	LAMI.RD305.D25.C2	LAMI.RD30

In [57]:
# Load the distance matrix artifact
table =q2.Artifact.load('/Users/bpessemier/Skin_microbiome_projects_analysis/L_oreal_acne/data/16S/174950_16S_taxonomy_collapsed_absolute/174950_rarefied_table_unannotated_absolute_abundances_T.qza')
data = table.view(pd.DataFrame)
print(data)

                   GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT  \
LAMI.RD001.D0.C1                                                11.0                                                                                                        
LAMI.RD001.D0.C2                                                 7.0                                                                                                        
LAMI.RD001.D14.C1                                                0.0                                                                                                        
LAMI.RD001.D14.C2                                                3.0                                                                                                        
LAMI.RD001.D28.C1                                                0.0                                                                   

In [34]:
# Load the distance matrix artifact
#table =q2.Artifact.load('/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Data/16S/CTF/filtered_174950.qza')
table =q2.Artifact.load('/Users/bpessemier/Skin_microbiome_projects_analysis/Acne_Longitudinal_Microbiome/Data/16S/CTF/filtered_174950_T.qza')
#
data = table.view(pd.DataFrame)
print(data)

                   GATGAACGCTGGCGGCGTGCTTAACACATGCAAGTCGAACGGAAAGGCCACAGCTTGCTGTGGTACTCGAGTGGCGAACGGGTGAGTAACACGTGGGTGATCTGCCCTGTACTTCGGGATAAGCTTGGGAAACTGGGTCTAATACCGGAT  \
LAMI.RD001.D0.C1                                                11.0                                                                                                        
LAMI.RD001.D0.C2                                                 7.0                                                                                                        
LAMI.RD001.D14.C1                                                0.0                                                                                                        
LAMI.RD001.D14.C2                                                3.0                                                                                                        
LAMI.RD001.D28.C1                                                0.0                                                                   

In [69]:
# Transpose the filtered biom table
data_T = data.transpose()

# Convert the transposed BIOM table to a QIIME 2 Artifact
data_artifact = q2.Artifact.import_data('FeatureTable[Frequency]', data_T)
    
# Save the artifact as a .qza file
data_artifact.save(f'../Data/16S/CTF/filtered_174950_T.qza')

'../Data/16S/CTF/filtered_174950_T.qza'

In [31]:
# Load feature by sample table for abundance data
table = Artifact.load(f'../Data/16S/CTF/filtered_174951_T.qza')
data = table.view(pd.DataFrame)
print(data)

                   TACGTAGGGTGCGAGCGTTATCCGGAATTATTGGGCGTAAAGAGCTCGTAGGCGGTTTGTCGCGTCTGCCGTGAAAGTCCGGGGCTTAACTCCGGATCTGCGGTGGGTACGGGCAGACTAGAGTGCAGTAGGGGAGACTGGAATTCCTGG  \
LAMI.RD001.D0.C1                                                 0.0                                                                                                        
LAMI.RD001.D14.C1                                                0.0                                                                                                        
LAMI.RD004.D0.C2                                                 0.0                                                                                                        
LAMI.RD001.D0.C2                                                 0.0                                                                                                        
LAMI.RD004.D28.C1                                                1.0                                                                   